# LeetCode 568 — Maximum Vacation Days bằng Python

- Link: https://leetcode.com/problems/maximum-vacation-days/
- Độ khó: **Hard** (bài **Premium**)
- Chủ đề: Quy hoạch động (DP) trên đồ thị theo thời gian, Ma trận kề, Đệ quy có nhớ

> **Ghi chú Premium:** đây là bài dành cho tài khoản Premium. Đề bài dưới đây được viết lại theo hiểu biết về bài (đề gốc, ví dụ và ràng buộc phổ biến). Nếu bạn có tài khoản Premium, hãy đối chiếu lại câu chữ, ví dụ và ràng buộc trên LeetCode.

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxVacationDays(self, flights: List[List[int]], days: List[List[int]]) -> int:

```

## 1. Giới thiệu & mục tiêu học

Bài này là ví dụ chuẩn của **DP theo "lớp thời gian"**: mỗi tuần là một lớp, mỗi thành phố là một đỉnh, chuyến bay là cạnh nối lớp này sang lớp sau. Bài toán trở thành **đường đi có tổng trọng số lớn nhất** qua `k` lớp.

Sau notebook này bạn sẽ:

1. Mô hình hóa bài toán thành **đồ thị nhiều lớp (layered graph)** theo tuần.
2. Viết vét cạn DFS, rồi nhận ra **bài toán con lặp lại** `(tuần, thành phố)` → đệ quy có nhớ.
3. Viết DP **xuôi** (forward, cần đánh dấu "không tới được") và DP **ngược** (backward, không cần đánh dấu).
4. Tối ưu bằng **danh sách kề** và biết **truy vết** lộ trình tối ưu.
5. Kiểm chứng mọi cách bằng brute force trên dữ liệu ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Công ty cho bạn đi công tác kết hợp nghỉ phép qua `n` thành phố (đánh số `0..n-1`) trong `k` tuần. Quy tắc:

1. Bạn bắt đầu ở **thành phố 0** vào **sáng thứ Hai của tuần 0**.
2. Ma trận `flights` kích thước `n × n`: `flights[i][j] = 1` nghĩa là có chuyến bay từ `i` sang `j`; `flights[i][i] = 0` (nhưng bạn luôn được **ở lại** thành phố hiện tại).
3. Mỗi tuần bạn chỉ được bay **tối đa một lần**, vào **sáng thứ Hai** (thời gian bay coi như không đáng kể). Tức là đầu mỗi tuần (kể cả tuần 0) bạn chọn: ở lại, hoặc bay sang một thành phố có chuyến bay trực tiếp.
4. Ma trận `days` kích thước `n × k`: `days[i][w]` = số ngày nghỉ **tối đa** bạn được hưởng nếu ở thành phố `i` trong tuần `w`.
5. Nếu bay từ `A` sang `B` vào thứ Hai tuần `w`, ngày nghỉ tuần đó tính theo `B`.

Trả về **tổng số ngày nghỉ lớn nhất** trong `k` tuần.

**Ví dụ 1**

```text
Input:  flights = [[0,1,1],[1,0,1],[1,1,0]], days = [[1,3,1],[6,0,3],[3,3,3]]
Output: 12
Giải thích: Một lộ trình tốt nhất:
  Tuần 0: bay 0 -> 1, nghỉ 6 ngày.
  Tuần 1: bay 1 -> 2, nghỉ 3 ngày.
  Tuần 2: ở lại 2, nghỉ 3 ngày.
  Tổng 6 + 3 + 3 = 12.
```

**Ví dụ 2**

```text
Input:  flights = [[0,0,0],[0,0,0],[0,0,0]], days = [[1,1,1],[7,7,7],[7,7,7]]
Output: 3
Giải thích: không có chuyến bay nào, phải ở thành phố 0 cả 3 tuần: 1 + 1 + 1.
```

**Ví dụ 3**

```text
Input:  flights = [[0,1,1],[1,0,1],[1,1,0]], days = [[7,0,0],[0,7,0],[0,0,7]]
Output: 21
Giải thích: tuần 0 ở 0 (7), tuần 1 bay sang 1 (7), tuần 2 bay sang 2 (7).
```

**Ràng buộc (theo đề phổ biến)**

- `n == flights.length == flights[i].length == days.length`
- `k == days[i].length`
- `1 <= n, k <= 100`
- `flights[i][j]` là `0` hoặc `1`, `flights[i][i] == 0`
- `0 <= days[i][j] <= 7`

In [1]:
flights1 = [[0, 1, 1], [1, 0, 1], [1, 1, 0]]
days1 = [[1, 3, 1], [6, 0, 3], [3, 3, 3]]

def route_value(route, flights, days):
    """route[w] = thành phố ở trong tuần w. Kiểm tra hợp lệ và tính tổng ngày nghỉ."""
    cur = 0
    total = 0
    for w, city in enumerate(route):
        if city != cur and not flights[cur][city]:
            return None                       # không có chuyến bay
        cur = city
        total += days[city][w]
    return total

print(route_value([1, 2, 2], flights1, days1))   # lộ trình trong ví dụ 1
print(route_value([0, 0, 0], flights1, days1))   # ở nhà cả 3 tuần

12
5


## 3. Kiến thức nền

### 3.1. Ma trận kề và danh sách kề

- **Ma trận kề** `flights[i][j]`: kiểm tra "có cạnh `i → j` không" trong `O(1)`, nhưng duyệt mọi hàng xóm của `i` tốn `O(n)`.
- **Danh sách kề** `adj[i]` = các `j` có cạnh: duyệt hàng xóm chỉ tốn `O(bậc của i)`.

In [2]:
adj = [[j for j in range(3) if flights1[i][j]] for i in range(3)]
print("danh sách kề:", adj)
flights_sparse = [[0, 1, 0], [0, 0, 0], [0, 0, 0]]
print("đồ thị thưa:", [[j for j in range(3) if flights_sparse[i][j]] for i in range(3)])

danh sách kề: [[1, 2], [0, 2], [0, 1]]
đồ thị thưa: [[1], [], []]


### 3.2. "Âm vô cùng" để đánh dấu trạng thái không tới được

Trong DP lấy max, trạng thái không thể đạt tới nên mang giá trị `-inf` (`float('-inf')`) để nó không bao giờ thắng phép `max`. Cộng `-inf` với số hữu hạn vẫn là `-inf`.

In [3]:
NEG = float('-inf')
print(max(NEG, 5), NEG + 7, max([NEG, NEG]))
# Hoặc dùng một số âm đủ lớn khi muốn giữ kiểu int
NEG_INT = -10**9
print(max(NEG_INT, 0))

5 -inf -inf
0


### 3.3. Đệ quy có nhớ với `functools.lru_cache`

Hàm `best(w, c)` = "điểm tốt nhất từ tuần `w` khi đang ở `c`" được gọi lại nhiều lần với cùng `(w, c)`; `@lru_cache` lưu kết quả để mỗi cặp chỉ tính một lần.

In [4]:
from functools import lru_cache

@lru_cache(maxsize=None)
def grid_paths(r, c):
    """Số đường đi từ (0,0) tới (r,c) chỉ đi phải/xuống — ví dụ memo kinh điển."""
    if r == 0 or c == 0:
        return 1
    return grid_paths(r - 1, c) + grid_paths(r, c - 1)

print(grid_paths(10, 10), grid_paths.cache_info())

184756 CacheInfo(hits=81, misses=120, maxsize=None, currsize=120)


### 3.4. DP "cuốn chiếu" — chỉ giữ một hàng

Khi trạng thái tuần `w+1` chỉ phụ thuộc tuần `w`, ta giữ **hai mảng** `cur`, `nxt` (hoặc tạo mảng mới mỗi vòng) thay vì cả bảng `k × n`.

In [5]:
cur = [1, 0, 0]
for step in range(3):
    nxt = [cur[0] + cur[2], cur[0] + cur[1], cur[1] + cur[2]]   # ví dụ chuyển trạng thái bất kỳ
    cur = nxt
    print(step, cur)

0 [1, 1, 0]
1 [1, 2, 1]
2 [2, 3, 3]


### 3.5. Hàm `max` với generator và `default`

`max(..., default=x)` trả `x` khi dãy rỗng — tiện khi một thành phố không có chuyến bay nào.

In [6]:
print(max((v for v in []), default=-1))
print(max((v * v for v in [3, -5, 2]), default=-1))

-1
25


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Đồ thị nhiều lớp

Vẽ `k` cột (mỗi cột là một tuần), mỗi cột có `n` ô (thành phố). Từ ô `(w, i)` có mũi tên sang ô `(w+1, j)` nếu `j = i` (ở lại) hoặc `flights[i][j] = 1`. Ô `(w, j)` có trọng số `days[j][w]`. Thêm một "cột -1" chỉ có ô thành phố 0 (trước tuần 0) để mô tả việc tuần 0 cũng được bay.

Bài toán = **đường đi qua `k` cột có tổng trọng số lớn nhất**, xuất phát từ thành phố 0.

### 4.2. Tính tay ví dụ 1

`days = [[1,3,1],[6,0,3],[3,3,3]]`, mọi cặp thành phố đều có chuyến bay (đồ thị đầy đủ).

Gọi `dp[w][c]` = tổng ngày nghỉ lớn nhất **tính tới hết tuần `w`** và đang ở `c`.

- Tuần 0 (từ thành phố 0 bay đi đâu cũng được): `dp[0] = [1, 6, 3]`.
- Tuần 1: mỗi thành phố đều tới được từ mọi thành phố → lấy `max(dp[0]) = 6` cộng `days[c][1]`: `dp[1] = [6+3, 6+0, 6+3] = [9, 6, 9]`.
- Tuần 2: `max(dp[1]) = 9` → `dp[2] = [9+1, 9+3, 9+3] = [10, 12, 12]`.
- Đáp án `max(dp[2]) = 12` ✓.

### 4.3. Tính tay một đồ thị thưa

`flights = [[0,1,0],[0,0,1],[0,0,0]]` (chỉ `0 → 1`, `1 → 2`), `days = [[2,2,2],[1,5,1],[0,0,9]]`, `k = 3`.

- Tuần 0: ở 0 (2) hoặc bay sang 1 (1). Thành phố 2 **chưa tới được**: `dp[0] = [2, 1, -inf]`.
- Tuần 1: `dp[1][0] = 2 + 2 = 4` (chỉ từ 0); `dp[1][1] = max(2, 1) + 5 = 7` (từ 0 bay hoặc ở lại 1); `dp[1][2] = 1 + 0 = 1` (từ 1 bay). → `[4, 7, 1]`.
- Tuần 2: `dp[2][0] = 4 + 2 = 6`; `dp[2][1] = max(4, 7) + 1 = 8`; `dp[2][2] = max(7, 1) + 9 = 16`. → `[6, 8, 16]`.
- Đáp án 16: lộ trình `1 → 1 → 2` (1 + 5 + 9 = 15)? Không: `dp[2][2] = dp[1][1] + 9 = 7 + 9`, và `dp[1][1] = dp[0][0] + 5 = 2 + 5` → lộ trình `0 → 1 → 2`: 2 + 5 + 9 = **16** ✓.

In [7]:
flights2 = [[0, 1, 0], [0, 0, 1], [0, 0, 0]]
days2 = [[2, 2, 2], [1, 5, 1], [0, 0, 9]]
print(route_value([0, 1, 2], flights2, days2), route_value([1, 1, 2], flights2, days2), route_value([2, 2, 2], flights2, days2))

16 15 None


## 5. Cách 1 — Brute Force: DFS thử mọi lộ trình

**Ý tưởng.** Ở mỗi tuần, thử mọi thành phố có thể tới (ở lại hoặc bay), đệ quy sang tuần sau, lấy max. Đây là liệt kê **mọi lộ trình hợp lệ**.

In [8]:
def vacation_brute(flights, days):
    n, k = len(days), len(days[0])

    def dfs(w, city):
        """Tổng ngày nghỉ lớn nhất từ tuần w trở đi, khi đang ở city vào sáng thứ Hai tuần w (trước khi bay)."""
        if w == k:
            return 0
        best = 0
        for nxt in range(n):
            if nxt == city or flights[city][nxt]:
                best = max(best, days[nxt][w] + dfs(w + 1, nxt))
        return best

    return dfs(0, 0)

flights3 = [[0, 1, 1], [1, 0, 1], [1, 1, 0]]
days3 = [[7, 0, 0], [0, 7, 0], [0, 0, 7]]
print(vacation_brute(flights1, days1), vacation_brute([[0] * 3 for _ in range(3)], [[1, 1, 1], [7, 7, 7], [7, 7, 7]]),
      vacation_brute(flights3, days3), vacation_brute(flights2, days2))
assert vacation_brute(flights1, days1) == 12
assert vacation_brute([[0] * 3 for _ in range(3)], [[1, 1, 1], [7, 7, 7], [7, 7, 7]]) == 3
assert vacation_brute(flights3, days3) == 21
assert vacation_brute(flights2, days2) == 16

12 3 21 16


**Độ phức tạp:** mỗi tuần có tới `n` lựa chọn → `O(n^k)` lộ trình. Với `n = k = 100` là `100^100` — không tưởng. Space `O(k)` cho stack đệ quy.

**Nút thắt:** `dfs(w, city)` chỉ phụ thuộc **hai số** `w` và `city`, nhưng được gọi lại vô số lần qua các lộ trình khác nhau. Chỉ có `k × n` cặp khác nhau!

In [9]:
import time, random

def count_calls(flights, days):
    n, k = len(days), len(days[0])
    calls = 0
    def dfs(w, city):
        nonlocal calls
        calls += 1
        if w == k:
            return 0
        return max(days[nx][w] + dfs(w + 1, nx) for nx in range(n) if nx == city or flights[city][nx])
    dfs(0, 0)
    return calls

full = lambda n: [[int(i != j) for j in range(n)] for i in range(n)]
for n, k in [(3, 6), (4, 7), (5, 8)]:
    d = [[1] * k for _ in range(n)]
    print(f"n={n}, k={k}: số lời gọi brute = {count_calls(full(n), d):>8}, số cặp (w, city) khác nhau = {(k + 1) * n}")

n=3, k=6: số lời gọi brute =     1093, số cặp (w, city) khác nhau = 21
n=4, k=7: số lời gọi brute =    21845, số cặp (w, city) khác nhau = 32
n=5, k=8: số lời gọi brute =   488281, số cặp (w, city) khác nhau = 45


## 6. Bước đột phá

**Insight (bài toán con lặp lại + không phụ thuộc quá khứ):** Tổng ngày nghỉ tốt nhất **từ tuần `w` trở đi** chỉ phụ thuộc bạn **đang ở đâu** lúc đầu tuần `w`, không phụ thuộc bạn đã tới đó bằng cách nào. Đây là tính chất **Markov** — điều kiện để dùng DP.

Hai cách định nghĩa:

- **Ngược (backward):** `best[w][c]` = tổng lớn nhất từ tuần `w` đến hết, khi sáng thứ Hai tuần `w` đang ở `c` (trước khi bay).

  ```text
  best[k][c] = 0
  best[w][c] = max over j in {c} ∪ adj[c} of ( days[j][w] + best[w+1][j] )
  Đáp án = best[0][0]
  ```

- **Xuôi (forward):** `dp[w][c]` = tổng lớn nhất từ tuần 0 tới hết tuần `w`, kết thúc tuần `w` ở `c`.

  ```text
  dp[-1] = [0, -inf, -inf, ...]           (trước tuần 0 chỉ ở thành phố 0)
  dp[w][j] = days[j][w] + max over i có thể tới j (i == j hoặc flights[i][j]) of dp[w-1][i]
  Đáp án = max(dp[k-1])
  ```

**Quy tắc:** nếu một thành phố **chưa thể tới được** ở tuần `w` (DP xuôi) thì giá trị của nó phải là `-inf`, nếu không nó sẽ "giả vờ" là điểm xuất phát hợp lệ.

## 7. Các cách tiếp theo

Notebook trình bày **4 cách**:

| # | Cách | Hướng |
|---|---|---|
| 1 | Brute force DFS (mục 5) | thử mọi lộ trình |
| 2 | Đệ quy có nhớ `best(w, c)` | ngược, top-down |
| 3 | DP xuôi bảng / cuốn chiếu | xuôi, bottom-up, cần `-inf` |
| 4 | DP ngược + danh sách kề | ngược, bottom-up, nhanh khi đồ thị thưa |

### 7.1. Cách 2 — Đệ quy có nhớ

**Giải thích.** Thêm `@lru_cache` vào DFS của Cách 1. Mỗi cặp `(w, city)` chỉ tính một lần, mỗi lần duyệt `n` thành phố đích.

In [10]:
def vacation_memo(flights, days):
    n, k = len(days), len(days[0])

    @lru_cache(maxsize=None)
    def best(w, city):
        if w == k:
            return 0
        res = 0
        for nxt in range(n):
            if nxt == city or flights[city][nxt]:
                res = max(res, days[nxt][w] + best(w + 1, nxt))
        return res

    ans = best(0, 0)
    return ans, best.cache_info().currsize

print(vacation_memo(flights1, days1), vacation_memo(flights2, days2))

(12, 10) (16, 9)


**Đọc code:**

- `best(w, city)`: đang ở `city` sáng thứ Hai tuần `w`, chưa bay.
- `nxt == city` là "ở lại" — luôn được phép (vì `flights[i][i] = 0` nên phải kiểm tra riêng).
- `days[nxt][w]`: ngày nghỉ tuần `w` tính theo thành phố **đích**.
- Số trạng thái lưu tối đa `(k+1) · n`.

**Code nộp LeetCode (Cách 2):** với `k = 100`, đệ quy sâu tối đa ~100 tầng — an toàn.

In [11]:
from typing import List

class SolutionMemo:
    def maxVacationDays(self, flights: List[List[int]], days: List[List[int]]) -> int:
        n, k = len(days), len(days[0])

        @lru_cache(maxsize=None)
        def best(w: int, city: int) -> int:
            if w == k:
                return 0
            res = 0
            for nxt in range(n):
                if nxt == city or flights[city][nxt]:
                    res = max(res, days[nxt][w] + best(w + 1, nxt))
            return res

        return best(0, 0)

assert SolutionMemo().maxVacationDays(flights1, days1) == 12
assert SolutionMemo().maxVacationDays(flights3, days3) == 21
print("SolutionMemo OK")

SolutionMemo OK


**Độ phức tạp Cách 2:** Time `O(k · n^2)`, Space `O(k · n)` cho cache.

### 7.2. Cách 3 — DP xuôi (bottom-up) với `-inf`

**Giải thích.** Đi theo thời gian thật: biết điểm tốt nhất tới hết tuần `w-1` ở mỗi thành phố, suy ra tuần `w`. Ban đầu chỉ thành phố 0 "đang có mặt" (giá trị 0), các thành phố khác `-inf`.

Ta viết bản **bảng đầy đủ** trước (để in và truy vết), sau đó bản cuốn chiếu.

In [12]:
NEG = float('-inf')

def vacation_forward_table(flights, days):
    n, k = len(days), len(days[0])
    prev = [0] + [NEG] * (n - 1)              # trước tuần 0: chỉ ở thành phố 0
    table = []
    for w in range(k):
        cur = [NEG] * n
        for j in range(n):                    # thành phố đích tuần w
            for i in range(n):                # thành phố cuối tuần w-1
                if prev[i] != NEG and (i == j or flights[i][j]):
                    cur[j] = max(cur[j], prev[i] + days[j][w])
        table.append(cur)
        prev = cur
    return max(prev), table

ans, table = vacation_forward_table(flights2, days2)
print("đáp án:", ans)
for w, row in enumerate(table):
    print(f"tuần {w}: {row}")

đáp án: 16
tuần 0: [2, 1, -inf]
tuần 1: [4, 7, 1]
tuần 2: [6, 8, 16]


**Đọc code:**

- `prev = [0] + [NEG]*(n-1)`: "cột -1" chỉ có thành phố 0.
- Hai vòng `j`, `i`: thành phố `j` tuần `w` nhận từ mọi `i` có thể bay tới `j` hoặc `i == j`.
- `prev[i] != NEG`: bỏ qua thành phố chưa tới được (thật ra `NEG + số` vẫn là `NEG` nên điều kiện này chỉ để rõ ý).
- Đáp án: `max(prev)` — kết thúc ở đâu cũng được. Luôn hữu hạn vì thành phố 0 luôn tới được (ở lại).

**Code nộp LeetCode (Cách 3, cuốn chiếu):** duyệt theo cạnh `i → j` thay vì theo `j` rồi `i` cũng được; ở đây dùng số nguyên âm lớn thay cho `-inf`.

In [13]:
class SolutionForward:
    def maxVacationDays(self, flights: List[List[int]], days: List[List[int]]) -> int:
        n, k = len(days), len(days[0])
        NEG = -10**9
        dp = [NEG] * n
        dp[0] = 0
        for w in range(k):
            new = [NEG] * n
            for i in range(n):
                if dp[i] == NEG:
                    continue
                for j in range(n):
                    if i == j or flights[i][j]:
                        v = dp[i] + days[j][w]
                        if v > new[j]:
                            new[j] = v
            dp = new
        return max(dp)

assert SolutionForward().maxVacationDays(flights1, days1) == 12
assert SolutionForward().maxVacationDays(flights2, days2) == 16
print("SolutionForward OK")

SolutionForward OK


**Độ phức tạp Cách 3:** Time `O(k · n^2)`, Space `O(n)` (cuốn chiếu).

### 7.3. Cách 4 — DP ngược + danh sách kề

**Giải thích.** Tính `best[w][c]` từ tuần cuối về tuần 0. Ưu điểm:

- **Không cần `-inf`**: mọi thành phố đều là "điểm xuất phát giả định" hợp lệ; ta chỉ lấy `best[0][0]` ở cuối, nên các thành phố không tới được không ảnh hưởng.
- Dùng **danh sách kề** `adj[c]` (cộng thêm chính `c`): nếu đồ thị thưa, mỗi bước chỉ tốn `O(bậc)`.

In [14]:
def vacation_backward(flights, days):
    n, k = len(days), len(days[0])
    adj = [[c] + [j for j in range(n) if flights[c][j]] for c in range(n)]   # gồm cả "ở lại"
    nxt = [0] * n                           # best[k][*] = 0
    for w in range(k - 1, -1, -1):
        cur = [0] * n
        for c in range(n):
            cur[c] = max(days[j][w] + nxt[j] for j in adj[c])
        nxt = cur
    return nxt[0]

print(vacation_backward(flights1, days1), vacation_backward(flights2, days2), vacation_backward(flights3, days3))
assert vacation_backward(flights2, days2) == 16

12 16 21


**Đọc code:**

- `adj[c] = [c] + ...`: luôn có lựa chọn ở lại, nên `max(...)` không bao giờ gặp dãy rỗng.
- Vòng `w` chạy **ngược** từ `k-1` về `0`: `cur` là `best[w]`, `nxt` là `best[w+1]`.
- Kết quả `nxt[0]` sau vòng lặp chính là `best[0][0]`.

**Code nộp LeetCode (Cách 4):**

In [15]:
class SolutionBackward:
    def maxVacationDays(self, flights: List[List[int]], days: List[List[int]]) -> int:
        n, k = len(days), len(days[0])
        adj = [[c] + [j for j in range(n) if flights[c][j]] for c in range(n)]
        best = [0] * n
        for w in range(k - 1, -1, -1):
            best = [max(days[j][w] + best[j] for j in adj[c]) for c in range(n)]
        return best[0]

assert SolutionBackward().maxVacationDays(flights1, days1) == 12
assert SolutionBackward().maxVacationDays([[0] * 3 for _ in range(3)], [[1, 1, 1], [7, 7, 7], [7, 7, 7]]) == 3
print("SolutionBackward OK")

SolutionBackward OK


Chú ý dòng `best = [ ... for c in range(n)]`: list comprehension tạo **list mới** từ `best` **cũ** rồi mới gán — nên không bị lỗi "dùng giá trị vừa cập nhật".

**Độ phức tạp Cách 4:** Time `O(k · (n + E))` với `E` = số chuyến bay (≤ `n^2`), cộng `O(n^2)` dựng `adj`; Space `O(n + E)`.

## 8. TRACE từng bước

### 8.1. Bảng DP xuôi cho đồ thị thưa ở mục 4.3

`flights2`: chỉ `0 → 1`, `1 → 2`. `days2 = [[2,2,2],[1,5,1],[0,0,9]]`.

| tuần | TP 0 | TP 1 | TP 2 | Giải thích ngắn |
|---|---|---|---|---|
| (trước) | 0 | -inf | -inf | chỉ đang ở 0 |
| 0 | 2 | 1 | -inf | 0 ở lại (2) / 0→1 (1); 2 chưa tới được |
| 1 | 4 | 7 | 1 | TP1 = max(2,1)+5 |
| 2 | 6 | 8 | 16 | TP2 = max(TP1=7, TP2=1)+9 |

In [16]:
def trace_forward(flights, days):
    n, k = len(days), len(days[0])
    prev = [0] + [NEG] * (n - 1)
    parent = []
    fmt = lambda row: "[" + ", ".join(f"{v:>4}" if v != NEG else "-inf" for v in row) + "]"
    print(f"trước : {fmt(prev)}")
    for w in range(k):
        cur, par = [NEG] * n, [None] * n
        for j in range(n):
            for i in range(n):
                if prev[i] != NEG and (i == j or flights[i][j]) and prev[i] + days[j][w] > cur[j]:
                    cur[j], par[j] = prev[i] + days[j][w], i
        moves = ", ".join(f"{par[j]}->{j}" for j in range(n) if par[j] is not None)
        print(f"tuần {w}: {fmt(cur)}   (đến từ: {moves})")
        parent.append(par)
        prev = cur
    # truy vết lộ trình
    end = max(range(n), key=lambda c: prev[c])
    route = [end]
    for w in range(k - 1, 0, -1):
        route.append(parent[w][route[-1]])
    route.reverse()
    print("lộ trình tốt nhất:", route, "tổng =", prev[end])
    return prev[end], route

val, route = trace_forward(flights2, days2)
assert val == 16 and route_value(route, flights2, days2) == 16

trước : [   0, -inf, -inf]
tuần 0: [   2,    1, -inf]   (đến từ: 0->0, 0->1)
tuần 1: [   4,    7,    1]   (đến từ: 0->0, 0->1, 1->2)
tuần 2: [   6,    8,   16]   (đến từ: 0->0, 1->1, 1->2)
lộ trình tốt nhất: [0, 1, 2] tổng = 16


### 8.2. Trace DP ngược cho ví dụ 1

`best[w][c]` = tốt nhất từ tuần `w` khi đang ở `c`. Đồ thị đầy đủ nên `best[w][c] = max_j (days[j][w] + best[w+1][j])` giống nhau cho mọi `c`:

| w | `days[*][w] + best[w+1][*]` | best[w] |
|---|---|---|
| 2 | `[1, 3, 3]` | `[3, 3, 3]` |
| 1 | `[3+3, 0+3, 3+3] = [6, 3, 6]` | `[6, 6, 6]` |
| 0 | `[1+6, 6+6, 3+6] = [7, 12, 9]` | `[12, 12, 12]` |

In [17]:
def trace_backward(flights, days):
    n, k = len(days), len(days[0])
    adj = [[c] + [j for j in range(n) if flights[c][j]] for c in range(n)]
    best = [0] * n
    for w in range(k - 1, -1, -1):
        cand = [days[j][w] + best[j] for j in range(n)]
        new = []
        for c in range(n):
            j_star = max(adj[c], key=lambda j: days[j][w] + best[j])
            new.append(days[j_star][w] + best[j_star])
        print(f"w={w}: days+best = {cand} -> best[w] = {new}")
        best = new
    return best[0]

assert trace_backward(flights1, days1) == 12

w=2: days+best = [1, 3, 3] -> best[w] = [3, 3, 3]
w=1: days+best = [6, 3, 6] -> best[w] = [6, 6, 6]
w=0: days+best = [7, 12, 9] -> best[w] = [12, 12, 12]


## 9. Code nộp LeetCode

Chọn Cách 4 (DP ngược + danh sách kề): ngắn, không cần `-inf`, nhanh với đồ thị thưa.

In [18]:
from typing import List

class Solution:
    def maxVacationDays(self, flights: List[List[int]], days: List[List[int]]) -> int:
        n, k = len(days), len(days[0])
        # adj[c]: các thành phố có thể ở trong tuần tới nếu đang ở c (gồm cả ở lại)
        adj = [[c] + [j for j in range(n) if flights[c][j]] for c in range(n)]
        best = [0] * n                      # best[c] cho tuần w+1 (ban đầu: sau tuần cuối)
        for w in range(k - 1, -1, -1):
            best = [max(days[j][w] + best[j] for j in adj[c]) for c in range(n)]
        return best[0]                      # xuất phát ở thành phố 0, tuần 0

sol = Solution()
assert sol.maxVacationDays(flights1, days1) == 12
assert sol.maxVacationDays([[0] * 3 for _ in range(3)], [[1, 1, 1], [7, 7, 7], [7, 7, 7]]) == 3
assert sol.maxVacationDays(flights3, days3) == 21
print("Solution OK")

Solution OK


## 10. Vì sao thuật toán đúng

**Mệnh đề.** `best[w][c]` (Cách 4) bằng tổng ngày nghỉ lớn nhất có thể đạt được trong các tuần `w..k-1`, nếu sáng thứ Hai tuần `w` (trước khi bay) bạn đang ở `c`.

**Chứng minh bằng quy nạp ngược theo `w`.**
- `w = k`: không còn tuần nào, giá trị 0. Đúng.
- Giả sử đúng với `w + 1`. Ở tuần `w`, một lộ trình hợp lệ bắt đầu bằng việc chọn thành phố `j` cho tuần `w` (`j = c` hoặc có chuyến bay `c → j`), được `days[j][w]`, và phần còn lại là một lộ trình hợp lệ từ tuần `w + 1` xuất phát ở `j` — **bất kỳ** lộ trình nào như vậy đều ghép được (các ràng buộc chỉ nối hai tuần liên tiếp). Vậy tối ưu = `max_j (days[j][w] + tối ưu từ (w+1, j))` = `max_j (days[j][w] + best[w+1][j])` theo giả thiết quy nạp.

Đáp án = `best[0][0]` vì bạn bắt đầu ở thành phố 0 trước khi bay tuần 0.

**DP xuôi** đúng theo lập luận đối xứng: `dp[w][j]` = tối ưu của mọi lộ trình tuần `0..w` kết thúc ở `j`; giá trị `-inf` đảm bảo chỉ những lộ trình xuất phát từ thành phố 0 được tính.

## 11. Phân tích Time/Space

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Brute force | `O(n^k)` | `O(k)` | chỉ dùng kiểm chứng |
| 2. Đệ quy có nhớ | `O(k · n^2)` | `O(k · n)` | top-down |
| 3. DP xuôi cuốn chiếu | `O(k · n^2)` | `O(n)` | cần `-inf` |
| 4. DP ngược + adj | `O(n^2 + k · (n + E))` | `O(n + E)` | **nộp bài** |

Với `n = k = 100`: `k · n^2 = 10^6` phép — rất nhanh.

In [19]:
random.seed(0)
n = k = 100
F = [[int(i != j and random.random() < 0.5) for j in range(n)] for i in range(n)]
D = [[random.randint(0, 7) for _ in range(k)] for _ in range(n)]
for name, cls in [("Memo", SolutionMemo), ("Forward", SolutionForward), ("Backward", SolutionBackward)]:
    t = time.perf_counter()
    r = cls().maxVacationDays(F, D)
    print(f"{name:>8}: {r}  {time.perf_counter() - t:.3f}s")

    Memo: 700  0.087s
 Forward: 700  0.048s


Backward: 700  0.021s


## 12. Bộ test

In [20]:
impls = {
    "memo": lambda f, d: vacation_memo(f, d)[0],
    "forward_table": lambda f, d: vacation_forward_table(f, d)[0],
    "backward": vacation_backward,
    "SolutionMemo": lambda f, d: SolutionMemo().maxVacationDays(f, d),
    "SolutionForward": lambda f, d: SolutionForward().maxVacationDays(f, d),
    "SolutionBackward": lambda f, d: SolutionBackward().maxVacationDays(f, d),
    "Solution": lambda f, d: Solution().maxVacationDays(f, d),
}

# Ví dụ đề
cases = [
    (flights1, days1, 12),
    ([[0] * 3 for _ in range(3)], [[1, 1, 1], [7, 7, 7], [7, 7, 7]], 3),
    (flights3, days3, 21),
]
# Edge case
cases += [
    ([[0]], [[5]], 5),                                   # 1 thành phố, 1 tuần
    ([[0]], [[1, 2, 3, 4]], 10),                         # 1 thành phố, nhiều tuần
    ([[0, 1], [0, 0]], [[0, 0], [7, 7]], 14),            # bay ngay tuần 0
    ([[0, 0], [1, 0]], [[0, 0], [7, 7]], 0),             # chỉ có 1 -> 0, không tới được 1
    ([[0, 1], [1, 0]], [[0, 0, 0], [0, 0, 0]], 0),       # toàn số 0
    (flights2, days2, 16),
]
for f, d, exp in cases:
    for name, fn in impls.items():
        assert fn(f, d) == exp, (name, f, d, fn(f, d))
    assert vacation_brute(f, d) == exp
print("Ví dụ đề + edge case OK")

# Ngẫu nhiên có seed, đối chiếu brute force
random.seed(568)
for _ in range(300):
    n = random.randint(1, 4)
    k = random.randint(1, 5)
    p = random.random()
    f = [[int(i != j and random.random() < p) for j in range(n)] for i in range(n)]
    d = [[random.randint(0, 7) for _ in range(k)] for _ in range(n)]
    b = vacation_brute(f, d)
    for name, fn in impls.items():
        assert fn(f, d) == b, (name, f, d)
print("300 test ngẫu nhiên khớp brute force")

Ví dụ đề + edge case OK
300 test ngẫu nhiên khớp brute force


## 13. Lỗi người mới hay gặp

### Lỗi 1 — DP xuôi nhưng khởi tạo mọi thành phố bằng 0

In [21]:
def wrong_init_zero(flights, days):
    n, k = len(days), len(days[0])
    dp = [0] * n                      # SAI: coi như đang ở mọi thành phố
    for w in range(k):
        dp = [days[j][w] + max(dp[i] for i in range(n) if i == j or flights[i][j]) for j in range(n)]
    return max(dp)

f, d = [[0, 0], [1, 0]], [[0, 0], [7, 7]]
print("sai:", wrong_init_zero(f, d), "| đúng:", Solution().maxVacationDays(f, d))

sai: 14 | đúng: 0


**Vì sao sai:** thành phố 1 không tới được từ 0, nhưng khởi tạo 0 khiến DP tưởng ta đang ở 1 từ đầu → được 14 ngày "ảo".

**Code đúng:** `dp = [0] + [-inf] * (n - 1)` (Cách 3), hoặc dùng DP ngược và chỉ lấy `best[0]` (Cách 4).

### Lỗi 2 — Quên lựa chọn "ở lại"

In [22]:
def wrong_no_stay(flights, days):
    n, k = len(days), len(days[0])
    adj = [[j for j in range(n) if flights[c][j]] for c in range(n)]     # thiếu c
    best = [0] * n
    for w in range(k - 1, -1, -1):
        best = [max((days[j][w] + best[j] for j in adj[c]), default=-10**9) for c in range(n)]
    return best[0]

f, d = [[0, 0, 0]] * 3, [[1, 1, 1], [7, 7, 7], [7, 7, 7]]
print("sai:", wrong_no_stay(f, d), "| đúng:", Solution().maxVacationDays(f, d))

sai: -1000000000 | đúng: 3


**Vì sao sai:** `flights[i][i] = 0` nhưng đề cho phép ở lại. Bỏ quên lựa chọn này khiến ví dụ 2 (không có chuyến bay) cho kết quả vô nghĩa.

**Code đúng:** `adj[c] = [c] + [...]` hoặc điều kiện `j == c or flights[c][j]`.

### Lỗi 3 — Tính ngày nghỉ theo thành phố xuất phát thay vì thành phố đến

In [23]:
def wrong_origin_days(flights, days):
    n, k = len(days), len(days[0])
    @lru_cache(maxsize=None)
    def best(w, c):
        if w == k:
            return 0
        return max(days[c][w] + best(w + 1, j) for j in range(n) if j == c or flights[c][j])   # SAI: days[c]
    return best(0, 0)

print("sai:", wrong_origin_days(flights1, days1), "| đúng:", Solution().maxVacationDays(flights1, days1))

sai: 7 | đúng: 12


**Vì sao sai:** bay vào sáng thứ Hai nên cả tuần bạn ở thành phố **đến**. Dùng `days[c][w]` (thành phố cũ) làm lệch mọi thứ một tuần, và tuần 0 luôn bị tính ở thành phố 0.

**Code đúng:** `days[j][w] + best(w + 1, j)`.

### Lỗi 4 — Cập nhật DP tại chỗ trong cùng một tuần

In [24]:
def wrong_inplace(flights, days):
    n, k = len(days), len(days[0])
    NEG = -10**9
    dp = [0] + [NEG] * (n - 1)
    for w in range(k):
        for j in range(n):
            dp[j] = max(dp[i] for i in range(n) if i == j or flights[i][j]) + days[j][w]
    return max(dp)

f, d = [[0, 1, 0], [0, 0, 1], [0, 0, 0]], [[0, 0], [0, 0], [5, 5]]
print("sai:", wrong_inplace(f, d), "| đúng:", Solution().maxVacationDays(f, d))

sai: 10 | đúng: 5


**Vì sao sai:** khi tính `dp[2]` tuần 0, `dp[1]` đã là giá trị **tuần 0** (vừa bay 0 → 1) chứ không phải "trước tuần 0" → như thể bay hai chặng `0 → 1 → 2` trong một buổi sáng.

**Code đúng:** luôn tạo mảng mới `new` cho tuần `w` và chỉ đọc mảng tuần trước.

## 14. Mẹo & nhận dạng pattern

- **"Mỗi bước thời gian chọn một trạng thái trong `n` trạng thái, chuyển trạng thái theo đồ thị cố định, cộng điểm"** → DP `dp[thời gian][trạng thái]`, Time `O(T · n^2)` hoặc `O(T · (n + E))`.
- **DP ngược** thường gọn hơn khi điểm xuất phát cố định, điểm kết thúc tự do (không cần `-inf`). **DP xuôi** gọn hơn khi điểm kết thúc cố định.
- Đề nói "ở lại luôn được" mà ma trận có đường chéo 0 → nhớ **thêm cạnh tự vòng**.
- Muốn **in lộ trình** → lưu `parent` (DP xuôi) hoặc `choice` (DP ngược) rồi truy vết.
- Nếu `T` khổng lồ mà `n` nhỏ → phép nhân ma trận trên đại số `(max, +)` + lũy thừa nhanh.

## 15. Biến thể

1. **Giới hạn tổng số chuyến bay `F`:** thêm chiều "đã bay bao nhiêu lần" → `dp[w][c][f]` (Bài tập 5).
2. **Mỗi chuyến bay tốn chi phí:** trừ chi phí khi `j != c`.
3. **Bắt buộc kết thúc ở thành phố `t`:** DP xuôi, đáp án `dp[k-1][t]`.
4. **Cheapest Flights Within K Stops (LeetCode 787):** đồ thị theo lớp "số chặng".
5. **Knight Probability (LeetCode 688), Out of Boundary Paths (LeetCode 576):** cùng mô hình "thời gian × trạng thái".

In [25]:
# Biến thể 2: mỗi chuyến bay tốn cost[c][j] ngày (trừ vào tổng)
def vacation_with_cost(flights, days, cost):
    n, k = len(days), len(days[0])
    best = [0] * n
    for w in range(k - 1, -1, -1):
        best = [max(days[j][w] + best[j] - (cost[c][j] if j != c else 0)
                    for j in range(n) if j == c or flights[c][j]) for c in range(n)]
    return best[0]

cost = [[0, 4, 4], [4, 0, 4], [4, 4, 0]]
print("có phí bay:", vacation_with_cost(flights1, days1, cost), "| không phí:", vacation_with_cost(flights1, days1, [[0] * 3] * 3))
assert vacation_with_cost(flights1, days1, [[0] * 3] * 3) == 12

có phí bay: 5 | không phí: 12


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force DFS | thử mọi lộ trình | `O(n^k)` | `O(k)` | kiểm chứng `n, k` nhỏ |
| 2. Đệ quy có nhớ | `best(w, c)` + cache | `O(k · n^2)` | `O(k · n)` | viết nhanh từ brute force |
| 3. DP xuôi | `dp[w][j]`, khởi tạo `-inf` | `O(k · n^2)` | `O(n)` | khi cần đích cố định / truy vết theo thời gian |
| 4. DP ngược + adj | `best[w][c]` từ cuối về đầu | `O(n^2 + k(n + E))` | `O(n + E)` | **nộp bài**, đồ thị thưa |

## 17. Bài tập

**Bài 1 (đoán output).** `flights = [[0,1],[1,0]]`, `days = [[1,1,1],[0,7,0]]`. Không chạy code, đáp án bao nhiêu?

**Bài 2 (tính tay).** Với `flights = [[0,1,0],[0,0,1],[1,0,0]]` (vòng `0→1→2→0`), `days = [[3,0,3],[0,4,0],[5,0,6]]`, hãy lập bảng DP xuôi 3 tuần.

**Bài 3 (điền TODO).** Hoàn thiện `best_route(flights, days)` trả về `(tổng, lộ trình)` bằng DP ngược + lưu lựa chọn.

**Bài 4 (tự viết lại).** Viết lại DP xuôi **có dùng danh sách kề ngược** `radj[j]` = các `i` bay được tới `j` (gồm cả `j`).

**Bài 5 (mini challenge).** Thêm ràng buộc: tổng số chuyến bay **không quá `F`**. Viết `vacation_limited(flights, days, F)`.

In [26]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def best_route(flights, days):
    # TODO: DP ngược; choice[w][c] = thành phố j tốt nhất cho tuần w khi đang ở c
    #       sau đó đi xuôi từ c = 0 để dựng lộ trình
    pass

# Bài 4 — TODO
def vacation_forward_radj(flights, days):
    # TODO: radj[j] = [j] + [i for i in range(n) if flights[i][j]]
    pass

# Bài 5 — TODO
def vacation_limited(flights, days, F):
    # TODO: best[w][c][f] với f = số chuyến bay còn được dùng
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: tuần 1 thành phố 1 có 7 ngày.
- Mức 2: có thể ở 0 tuần 0, bay sang 1 tuần 1, bay về 0 tuần 2.
- Mức 3: 1 + 7 + 1.

**Bài 2.**
- Mức 1: trước tuần 0: `[0, -inf, -inf]`.
- Mức 2: tuần 0: ở 0 (3) hoặc bay 0→1 (0) → `[3, 0, -inf]`.
- Mức 3: tuần 1: `[3+0, max(3,0)+4, 0+0] = [3, 7, 0]`; tuần 2 tự tính (nhớ: TP 1 nhận từ TP 1 ở lại hoặc TP 0 bay sang).

**Bài 3.**
- Mức 1: giống Cách 4 nhưng giữ `choice`.
- Mức 2: `choice[w][c] = argmax_j (days[j][w] + best_next[j])`.
- Mức 3: `c = 0; for w in range(k): c = choice[w][c]; route.append(c)`.

**Bài 4.**
- Mức 1: `new[j] = days[j][w] + max(dp[i] for i in radj[j])`.
- Mức 2: khởi tạo `dp = [0] + [NEG]*(n-1)`.
- Mức 3: `NEG + x` vẫn rất âm, `max` không chọn.

**Bài 5.**
- Mức 1: ở lại không tốn lượt bay; bay tốn 1.
- Mức 2: `best[c][f] = max(days[c][w] + next[c][f], max_{j kề} days[j][w] + next[j][f-1] nếu f > 0)`.
- Mức 3: đáp án `best[0][F]`; đối chiếu brute force có đếm chuyến bay.

## 19. Đáp án tham khảo

**Bài 1:** 9. **Bài 2:** tuần 0 `[3, 0, -inf]`, tuần 1 `[3, 7, 0]`, tuần 2 `[max(3, 0)+3, max(7, 3)+0, max(0, 7)+6] = [6, 7, 13]` → đáp án 13 (lộ trình `0 → 1 → 2`: 3 + 4 + 6).

In [27]:
assert Solution().maxVacationDays([[0, 1], [1, 0]], [[1, 1, 1], [0, 7, 0]]) == 9
ans_b2, tab_b2 = vacation_forward_table([[0, 1, 0], [0, 0, 1], [1, 0, 0]], [[3, 0, 3], [0, 4, 0], [5, 0, 6]])
assert tab_b2 == [[3, 0, NEG], [3, 7, 0], [6, 7, 13]] and ans_b2 == 13

# Bài 3
def best_route(flights, days):
    n, k = len(days), len(days[0])
    best = [0] * n
    choice = [None] * k
    for w in range(k - 1, -1, -1):
        new, ch = [0] * n, [0] * n
        for c in range(n):
            opts = [c] + [j for j in range(n) if flights[c][j]]
            j = max(opts, key=lambda j: days[j][w] + best[j])
            new[c], ch[c] = days[j][w] + best[j], j
        best, choice[w] = new, ch
    route, c = [], 0
    for w in range(k):
        c = choice[w][c]
        route.append(c)
    return best[0], route

total, route = best_route(flights1, days1)
print(total, route)
assert total == 12 and route_value(route, flights1, days1) == 12

12 [1, 0, 1]


In [28]:
# Bài 4
def vacation_forward_radj(flights, days):
    n, k = len(days), len(days[0])
    NEG = -10**9
    radj = [[j] + [i for i in range(n) if flights[i][j]] for j in range(n)]
    dp = [0] + [NEG] * (n - 1)
    for w in range(k):
        dp = [days[j][w] + max(dp[i] for i in radj[j]) for j in range(n)]
    return max(dp)

# Bài 5
def vacation_limited(flights, days, F):
    n, k = len(days), len(days[0])
    nxt = [[0] * (F + 1) for _ in range(n)]
    for w in range(k - 1, -1, -1):
        cur = [[0] * (F + 1) for _ in range(n)]
        for c in range(n):
            for f in range(F + 1):
                v = days[c][w] + nxt[c][f]                       # ở lại
                if f > 0:
                    for j in range(n):
                        if flights[c][j]:
                            v = max(v, days[j][w] + nxt[j][f - 1])
                cur[c][f] = v
        nxt = cur
    return nxt[0][F]

def vacation_limited_brute(flights, days, F):
    n, k = len(days), len(days[0])
    def dfs(w, c, f):
        if w == k:
            return 0
        res = days[c][w] + dfs(w + 1, c, f)
        if f > 0:
            for j in range(n):
                if flights[c][j]:
                    res = max(res, days[j][w] + dfs(w + 1, j, f - 1))
        return res
    return dfs(0, 0, F)

random.seed(5)
for _ in range(200):
    n, k = random.randint(1, 4), random.randint(1, 5)
    f = [[int(i != j and random.random() < 0.6) for j in range(n)] for i in range(n)]
    d = [[random.randint(0, 7) for _ in range(k)] for _ in range(n)]
    assert vacation_forward_radj(f, d) == Solution().maxVacationDays(f, d)
    F = random.randint(0, k)
    assert vacation_limited(f, d, F) == vacation_limited_brute(f, d, F)
    assert vacation_limited(f, d, k) == Solution().maxVacationDays(f, d)   # F = k: không giới hạn thực sự
print("Bài 4, 5 OK")

Bài 4, 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "DP thời gian × trạng thái (ngược)":**

```python
adj = [[c] + [j for j in range(n) if edge[c][j]] for c in range(n)]   # gồm cả ở lại
best = [0] * n                                                         # sau bước cuối
for t in range(T - 1, -1, -1):
    best = [max(gain[j][t] + best[j] for j in adj[c]) for c in range(n)]
return best[start]
```

**Câu thần chú:** *"Tuần là lớp, thành phố là đỉnh; đi ngược từ tuần cuối, nhớ cho phép ở lại."*

**Checklist trước khi submit:**

- [ ] Có lựa chọn **ở lại** (`j == c`) dù `flights[c][c] = 0`.
- [ ] Ngày nghỉ tính theo thành phố **đến**: `days[j][w]`.
- [ ] Tuần 0 cũng được bay (xuất phát trước tuần 0 ở thành phố 0).
- [ ] DP xuôi: khởi tạo `-inf` cho thành phố khác 0. DP ngược: trả về `best[0]`.
- [ ] Mỗi tuần tạo mảng **mới**, không sửa tại chỗ.
- [ ] Thử ví dụ không có chuyến bay (đáp án = tổng hàng `days[0]`).
- [ ] Đối chiếu brute force với `n, k` nhỏ.